# Frame Extractor

Extract frames from video at configurable fps.

**Input:** original video or folder of clips from Scene_Clip_Extraction  
**Output:** JPEG frames → Google Drive

**Suggested fps for rugby closeup:**
| FPS | Per second | Use case |
|-----|------------|---------|
| 2   | 2 frames   | Exposure measurement |
| 3   | 3 frames   | Light annotation |
| 5   | 5 frames   | Rugby closeup — **recommended** |
| 10  | 10 frames  | Slow-mo / fast action |
| 0   | —          | Use `TOTAL_FRAMES` instead of fps |

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
from pathlib import Path

# ── Input: single video or folder of clips ────────────────────────────────────
# Choose one:
INPUT_VIDEO  = '/content/drive/MyDrive/bradford_bulls/videos/M05_white_1080p.mp4'
INPUT_FOLDER = ''   # leave empty to use INPUT_VIDEO
                    # or: '/content/drive/MyDrive/bradford_bulls/clips/M05_white_1080p'

# ── Sampling ──────────────────────────────────────────────────────────────────
EXTRACT_FPS    = 5       # frames/second — set 0 to use TOTAL_FRAMES
TOTAL_FRAMES   = 0       # if > 0: evenly sample N frames from video (ignores EXTRACT_FPS)

# ── Output root ───────────────────────────────────────────────────────────────
OUTPUT_ROOT  = Path('/content/drive/MyDrive/bradford_bulls/frames')
JPEG_QUALITY = 95

# ── Resolve input files + source name ────────────────────────────────────────
# Output structure:
#   INPUT_VIDEO  → OUTPUT_ROOT / video_stem / frames
#   INPUT_FOLDER → OUTPUT_ROOT / folder_stem / clip_stem / frames
if INPUT_FOLDER:
    video_files = sorted(Path(INPUT_FOLDER).glob('*.mp4'))
    source_name = Path(INPUT_FOLDER).stem   # original video name (folder is named after it)
    use_subdir  = True
else:
    video_files = [Path(INPUT_VIDEO)]
    source_name = Path(INPUT_VIDEO).stem
    use_subdir  = False                     # no clip sub-level for a single video

print(f'Input videos  : {len(video_files)}')
for v in video_files:
    print(f'  {v.name}')
print(f'\nSource name   : {source_name}')
print(f'Extract FPS   : {EXTRACT_FPS if EXTRACT_FPS else "auto"}')
print(f'Total frames  : {TOTAL_FRAMES if TOTAL_FRAMES else "from fps"}')
print(f'Output root   : {OUTPUT_ROOT / source_name}')

## 3. Preview — estimate frame count before running

In [ ]:
import cv2

total_estimate = 0
print(f'{"File":<40} {"Duration":>10} {"FPS":>6} {"Est. frames":>12}')
print('-' * 72)

for vf in video_files:
    cap = cv2.VideoCapture(str(vf))
    vid_fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration  = n_frames / vid_fps
    cap.release()

    if TOTAL_FRAMES > 0:
        est = TOTAL_FRAMES
    elif EXTRACT_FPS > 0:
        est = int(duration * EXTRACT_FPS)
    else:
        est = n_frames

    total_estimate += est
    dur_str = f'{duration/60:.1f} min'
    print(f'{vf.name:<40} {dur_str:>10} {vid_fps:>6.1f} {est:>12,}')

print('-' * 72)
print(f'{"TOTAL":<40} {"": >10} {"": >6} {total_estimate:>12,}')
print(f'\nEst. storage : ~{total_estimate * 0.3:.0f} MB  (300KB/frame avg)')
print(f'\nOutput structure:')
if use_subdir:
    print(f'  {OUTPUT_ROOT}/{source_name}/')
    print(f'    clip_001/ → frames')
    print(f'    clip_002/ → frames  ...')
else:
    print(f'  {OUTPUT_ROOT}/{source_name}/')
    print(f'    frames')

## 4. Extract frames

In [ ]:
import cv2
from tqdm.notebook import tqdm
from pathlib import Path

grand_total = 0

for vf in video_files:
    cap      = cv2.VideoCapture(str(vf))
    vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # compute frame indices to extract
    if TOTAL_FRAMES > 0:
        # evenly distribute TOTAL_FRAMES across the video
        step = max(1, n_total // TOTAL_FRAMES)
        indices = list(range(0, n_total, step))[:TOTAL_FRAMES]
    elif EXTRACT_FPS > 0:
        step = max(1, int(round(vid_fps / EXTRACT_FPS)))
        indices = list(range(0, n_total, step))
    else:
        indices = list(range(n_total))  # all frames

    # OUTPUT_ROOT / source_name / clip_stem  (folder mode)
    # OUTPUT_ROOT / source_name              (single video mode)
    if use_subdir:
        out_subdir = OUTPUT_ROOT / source_name / vf.stem
    else:
        out_subdir = OUTPUT_ROOT / source_name
    out_subdir.mkdir(parents=True, exist_ok=True)

    n_saved = 0
    for idx in tqdm(indices, desc=vf.name[:30]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        t_ms = int(idx / vid_fps * 1000)
        fname = out_subdir / f'{vf.stem}_{idx:07d}_t{t_ms:08d}ms.jpg'
        cv2.imwrite(str(fname), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        n_saved += 1

    cap.release()
    grand_total += n_saved
    print(f'  {vf.name}: {n_saved} frames → {out_subdir}')

print(f'\nTotal: {grand_total} frames saved → {OUTPUT_ROOT / source_name}')

## 5. Sample frame viewer

In [ ]:
import random, math
import matplotlib.pyplot as plt
import cv2

SAMPLE_N = 12   # number of frames to preview
COLS     = 4

search_dir = OUTPUT_ROOT / source_name
all_frames = sorted(search_dir.rglob('*.jpg'))
sample     = random.sample(all_frames, min(SAMPLE_N, len(all_frames)))
sample.sort()

n_rows = math.ceil(len(sample) / COLS)
fig, axes = plt.subplots(n_rows, COLS,
                         figsize=(COLS * 4, n_rows * 2.8),
                         squeeze=False)
axes = axes.flatten()

for i, fp in enumerate(sample):
    img = cv2.imread(str(fp))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(fp.parent.name + '/' + fp.name[-20:], fontsize=6)
    axes[i].axis('off')

for i in range(len(sample), len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Sample frames — {source_name}  ({len(all_frames)} total)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()